线性回归的简洁实现

通过使用深度学习框架来简洁地实现线性回归模型，生成数据集

In [10]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
# 生成数据集
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

调用框架中现有的API来读取数据

In [11]:
def load_array(data_arrays, batch_size, is_train=True):
    """
    构造一个PyTorch数据迭代器
    data_arrays：一个列表，包含特征和标签
    batch_size：批量大小
    is_train：一个布尔值，指定是否需要打乱数据
    """ 
    # 使用了 PyTorch 的 TensorDataset API,将输入的特征张量和标签张量打包组合成一个数据集对象
    dataset = data.TensorDataset(*data_arrays)

    # 接收刚才创建的 dataset，并根据设定的 batch_size 将数据划分成一个个小批次。并且随机打乱
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

# 测试与输出结果(获取第一批次的数据长什么样子)
next(iter(data_iter))

[tensor([[-0.4129, -1.1242],
         [ 0.5950, -0.7730],
         [-2.0457, -1.3828],
         [-0.1069,  0.5303],
         [ 0.4720, -0.5317],
         [ 0.5768,  1.8279],
         [ 0.5452,  0.9468],
         [-0.5108, -1.0798],
         [-0.8131,  0.9754],
         [ 0.5619,  0.6222]]),
 tensor([[ 7.1947],
         [ 8.0093],
         [ 4.8220],
         [ 2.1853],
         [ 6.9539],
         [-0.8395],
         [ 2.0722],
         [ 6.8629],
         [-0.7369],
         [ 3.2144]])]

使用框架预定义好的层

In [12]:
from torch import nn

# 指定输入和输出的特征数量,并且使用 Sequential 类来构造一个线性层
net = nn.Sequential(nn.Linear(2, 1))            

初始化参数模型

In [13]:
net[0].weight.data.normal_(0, 0.01)     # 权重参数初始化为均值为0、标准差为0.01的正态分布随机数
net[0].bias.data.fill_(0)               # 偏差参数初始化为0

tensor([0.])

计算均方误差使用的是MSELoss类，也称平方范数

In [14]:
loss = nn.MSELoss()  # 均方误差损失函数

实例化SGD实例

In [ ]:
# 最少传入两个参数，参数权重和学习率
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

训练过程代码与我们从零开始实现时所做的非常相似

In [17]:
num_epochs = 3
for epoch in range(num_epochs):
    # 拿到当前批次的特征和标签
    for X, y in data_iter:    
        # 计算损失函数  
        l = loss(net(X), y)
        # 清零梯度
        trainer.zero_grad()
        # 反向传播
        l.backward()
        # 更新参数
        trainer.step()
    # 打印输出当前 epoch 的损失函数值
    with torch.no_grad():
        train_l = loss(net(features), labels)
        print(f'epoch {epoch + 1}, loss {float(train_l):f}')

epoch 1, loss 0.000103
epoch 2, loss 0.000103
epoch 3, loss 0.000103
